# Clustering Comparison: SKATER vs Ward Hierarchical

This notebook:
1. Tests whether cluster distributions are statistically significantly different (Kruskal-Wallis + Dunn post-hoc, Moran's I on residuals)
2. Compares cluster validity indices (silhouette, Calinski-Harabasz, Davies-Bouldin) between methods
3. Sweeps k-NN neighbors (k=3..30) for SKATER and evaluates sensitivity

See `../docs/clustering_tradeoffs.md` for the theoretical background and literature review.

**Feature set:** AEP thresholds (action / flood / moderate / major), log₁₀ return period

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import rplot

plt.style.use('ryan')

from pathlib import Path
from shapely.geometry import Point
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import (
    silhouette_score, calinski_harabasz_score, davies_bouldin_score,
    adjusted_rand_score
)
from scipy.sparse.csgraph import connected_components
import scipy.stats as stats
import scikit_posthocs as sp
import joblib

import libpysal
from spopt.region import Skater
from esda.moran import Moran

## Configuration

In [ ]:
DATA_DIR   = Path("/home/ryan/data/flood_hazard")
GAGES2_DIR = Path("/home/ryan/data/usgs/GAGES_2/basinchar_and_report_sept_2011/spreadsheets-in-csv-format")
OUT_DIR    = Path(".")

K_NEIGHBORS       = 22
FLOOR             = 30
N_CLUSTERS        = 8
MAX_DISTURB_INDEX = 15
WINSOR            = 0.02
KNN_SWEEP_MIN     = 3
KNN_SWEEP_MAX     = 30
FORCE_RERUN       = False   # set True to bypass cache
SAVEFIG           = True

CONUS_EXTENT = [-125, -66, 24, 50]

def make_conus_ax(fig, pos=111, title=''):
    subplot_args = pos if isinstance(pos, tuple) else (pos,)
    ax = fig.add_subplot(*subplot_args, projection=ccrs.AlbersEqualArea(
        central_longitude=-96, central_latitude=37.5))
    ax.set_extent(CONUS_EXTENT, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND,      facecolor='#f5f5f0', zorder=0)
    ax.add_feature(cfeature.OCEAN,     facecolor='#c8e0f0', zorder=0)
    ax.add_feature(cfeature.LAKES,     facecolor='#c8e0f0', zorder=1, alpha=0.6)
    ax.add_feature(cfeature.STATES,    linewidth=0.4, zorder=2, edgecolor='gray')
    ax.add_feature(cfeature.COASTLINE, linewidth=0.6, zorder=3)
    ax.add_feature(cfeature.BORDERS,   linewidth=0.6, zorder=3)
    if title:
        ax.set_title(title)
    return ax

## Load shared data

In [ ]:
ffa  = pd.read_parquet(DATA_DIR / "ffa" / "flood_frequency.parquet")
meta = pd.read_parquet(DATA_DIR / "metadata" / "site_info.parquet")[["site_no", "latitude", "longitude"]]

gages2 = pd.read_csv(GAGES2_DIR / "conterm_bas_classif.txt", encoding="latin1")
gages2["site_no"] = gages2["STAID"].astype(str).str.zfill(8)

print(f"FFA records : {len(ffa):,}")
print(f"Site meta   : {len(meta):,}")
print(f"GAGES-2     : {len(gages2):,}")

---
## AEP Threshold Preprocessing

Identical to `skater_clustering.ipynb` Analysis 2: filter, transform to log₁₀ return period, winsorize 2%, RobustScale.

In [ ]:
AEP_COLS = ["action_aep", "flood_aep", "moderate_aep", "major_aep"]

df = (
    ffa[ffa.record_ok & ~ffa.degenerate_fit]
    [["site_no"] + AEP_COLS]
    .dropna(subset=AEP_COLS)
    .merge(meta, on="site_no")
    .merge(gages2[["site_no", "HYDRO_DISTURB_INDX"]], on="site_no", how="left")
)
df = df[
    df.HYDRO_DISTURB_INDX.notna() & (df.HYDRO_DISTURB_INDX <= MAX_DISTURB_INDEX)
].reset_index(drop=True)

print(f"AEP analysis: {len(df):,} sites")

In [ ]:
RP_FEATS  = ["action_rp", "flood_rp", "moderate_rp", "major_rp"]
RP_SCALED = [f + "_s" for f in RP_FEATS]

for aep_col, rp_col in zip(AEP_COLS, RP_FEATS):
    df[rp_col] = np.log10(1.0 / df[aep_col].clip(lower=1e-6))

df_clip = df.copy()
for col in RP_FEATS:
    lo, hi = df[col].quantile([WINSOR, 1 - WINSOR])
    df_clip[col] = df[col].clip(lo, hi)

scaler = RobustScaler()
X = scaler.fit_transform(df_clip[RP_FEATS])
for i, col in enumerate(RP_SCALED):
    df[col] = X[:, i]

print("Feature ranges (scaled):")
print(pd.DataFrame(X, columns=RP_SCALED).describe().loc[["min", "50%", "max"]].round(3).to_string())

In [ ]:
gdf = gpd.GeoDataFrame(
    df,
    geometry=[Point(lon, lat) for lon, lat in zip(df.longitude, df.latitude)]
)

w = libpysal.weights.KNN.from_dataframe(gdf, k=K_NEIGHBORS)
n_comp, _ = connected_components(w.sparse, directed=False)
print(f"Sites: {len(gdf):,}  |  k={K_NEIGHBORS}  |  Connected components: {n_comp}")

## Run SKATER and Ward at k=8 for comparison

In [ ]:
# Load saved SKATER labels (k=22, n=8) to avoid re-running the slow solve
skater_csv = OUT_DIR / "site_regions_skater_aep_8.csv"
skater_saved = pd.read_csv(skater_csv)
skater_saved["site_no"] = skater_saved["site_no"].astype(str).str.zfill(8)

# Merge onto df (inner join — only sites present in saved output)
df = df.merge(skater_saved[["site_no", "cluster"]].rename(columns={"cluster": "cl_skater"}),
              on="site_no", how="inner")
gdf = gpd.GeoDataFrame(
    df,
    geometry=[Point(lon, lat) for lon, lat in zip(df.longitude, df.latitude)]
)
w = libpysal.weights.KNN.from_dataframe(gdf, k=K_NEIGHBORS)

# Re-extract feature matrix for the merged subset
X = df[RP_SCALED].values

print(f"Sites after inner join with SKATER labels: {len(df):,}")
print("SKATER cluster sizes:")
print(df["cl_skater"].value_counts().sort_index().to_string())

In [ ]:
hc = AgglomerativeClustering(n_clusters=N_CLUSTERS, linkage='ward')
df["cl_ward"] = hc.fit_predict(X) + 1

print(f"Ward cluster sizes:")
print(df["cl_ward"].value_counts().sort_index().to_string())
print(f"\nMin: {df['cl_ward'].value_counts().min()}  Max: {df['cl_ward'].value_counts().max()}")

---
# Section 2: Statistical Significance of Clusters

Are the cluster distributions statistically significantly different from each other?

## 2.1 Kruskal-Wallis + Dunn post-hoc

For each AEP feature, test: are distributions across clusters significantly different?  
Kruskal-Wallis: non-parametric omnibus test (no normality assumption).  
Dunn post-hoc: pairwise comparisons with Bonferroni correction.

In [ ]:
kw_results = []
dunn_matrices = {}  # {(feature, method): p_matrix}

for method, cl_col in [("SKATER", "cl_skater"), ("Ward", "cl_ward")]:
    for feat in RP_FEATS:
        groups = [df.loc[df[cl_col] == c, feat].values
                  for c in sorted(df[cl_col].unique())]
        H, p = stats.kruskal(*groups)
        kw_results.append({
            "method": method, "feature": feat,
            "H": round(H, 2), "p": p, "reject_H0": p < 0.05
        })

        dunn_p = sp.posthoc_dunn(
            df, val_col=feat, group_col=cl_col, p_adjust='bonferroni'
        )
        dunn_matrices[(feat, method)] = dunn_p

kw_df = pd.DataFrame(kw_results)
print("Kruskal-Wallis results:")
print(kw_df.to_string(index=False))

In [ ]:
feat_labels = ["Action", "Flood", "Moderate", "Major"]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for row, (method, cl_col) in enumerate([("SKATER", "cl_skater"), ("Ward", "cl_ward")]):
    for col, (feat, label) in enumerate(zip(RP_FEATS, feat_labels)):
        ax = axes[row, col]
        pmat = dunn_matrices[(feat, method)]
        log_p = np.log10(pmat.values + 1e-300)
        im = ax.imshow(log_p, cmap='RdYlGn', vmin=-4, vmax=0, aspect='auto')
        ax.set_xticks(range(len(pmat.columns)))
        ax.set_yticks(range(len(pmat.index)))
        ax.set_xticklabels([f"C{c}" for c in pmat.columns], fontsize=7)
        ax.set_yticklabels([f"C{c}" for c in pmat.index], fontsize=7)
        ax.set_title(f"{method} — {label}", fontsize=8)

        for i in range(len(pmat.index)):
            for j in range(len(pmat.columns)):
                val = pmat.values[i, j]
                txt = "*" if val < 0.05 else ""
                ax.text(j, i, txt, ha='center', va='center', fontsize=8, color='black')

plt.colorbar(im, ax=axes[:, -1].tolist(), shrink=0.8, label='log₁₀(p-value Bonferroni)')
plt.suptitle('Dunn post-hoc pairwise p-values (* = p < 0.05)', fontsize=11)
plt.tight_layout()
if SAVEFIG:
    plt.savefig(OUT_DIR / 'dunn_pvalue_heatmaps.png', dpi=200, bbox_inches='tight')
plt.show()

## 2.2 Cluster Validity Indices

Compare SKATER vs Ward at k=8 on three complementary metrics.

In [ ]:
validity_rows = []
for method, cl_col in [("SKATER", "cl_skater"), ("Ward", "cl_ward")]:
    labels = df[cl_col].values
    validity_rows.append({
        "Method":              method,
        "Silhouette":          round(silhouette_score(X, labels), 4),
        "Calinski-Harabasz":   round(calinski_harabasz_score(X, labels), 1),
        "Davies-Bouldin":      round(davies_bouldin_score(X, labels), 4),
        "ARI vs SKATER":       round(adjusted_rand_score(df["cl_skater"].values, labels), 3),
        "Min cluster size":    int(df[cl_col].value_counts().min()),
        "Max cluster size":    int(df[cl_col].value_counts().max()),
    })

validity_df = pd.DataFrame(validity_rows).set_index("Method")
print("Cluster validity indices:")
print(validity_df.to_string())

## 2.3 Violin Plots — Feature Distributions per Cluster

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 9), sharey='col')

for row, (method, cl_col) in enumerate([("SKATER", "cl_skater"), ("Ward", "cl_ward")]):
    clusters = sorted(df[cl_col].unique())
    colors = rplot.cluster_cmap(len(clusters)).colors

    for col, (feat, label) in enumerate(zip(RP_FEATS, feat_labels)):
        ax = axes[row, col]
        data = [df.loc[df[cl_col] == c, feat].values for c in clusters]

        vp = ax.violinplot(data, positions=clusters, showmedians=True, widths=0.7)
        for body, color in zip(vp['bodies'], colors):
            body.set_facecolor(color)
            body.set_alpha(0.7)
        for part in ['cmedians', 'cbars', 'cmins', 'cmaxes']:
            vp[part].set_color('black')
            vp[part].set_linewidth(0.8)

        ax.set_xticks(clusters)
        ax.set_xticklabels([f"C{c}" for c in clusters], fontsize=7)
        ax.set_xlabel("Cluster")
        if col == 0:
            ax.set_ylabel(f"{method}\nlog₁₀(return period, yr)", fontsize=8)
        if row == 0:
            ax.set_title(label)

plt.suptitle(f"AEP feature distributions per cluster — SKATER (top) vs Ward (bottom)  k={N_CLUSTERS}",
             fontsize=10)
plt.tight_layout()
if SAVEFIG:
    plt.savefig(OUT_DIR / 'violin_comparison.png', dpi=200, bbox_inches='tight')
plt.show()

## 2.4 Moran's I on Cluster Residuals

**Motivation:** If Ward residuals (distance of each site's feature vector to its Ward centroid) exhibit significant positive spatial autocorrelation, it means geographically close sites share similar residual patterns — a signal that SKATER is designed to capture. If SKATER residuals show *lower* spatial autocorrelation, SKATER has successfully absorbed the geographic structure that Ward missed.

Residual = Euclidean distance of a site's scaled feature vector from its cluster centroid (scalar).

In [ ]:
moran_results = []
for method, cl_col in [("SKATER", "cl_skater"), ("Ward", "cl_ward")]:
    labels = df[cl_col].values
    centroids = np.array([
        X[labels == c].mean(axis=0) for c in sorted(np.unique(labels))
    ])   # shape: (k, n_features)
    # Map cluster label (1-indexed) to centroid index
    sorted_labels = sorted(np.unique(labels))
    label_to_idx = {c: i for i, c in enumerate(sorted_labels)}
    centroid_per_site = np.array([centroids[label_to_idx[c]] for c in labels])
    residuals = np.linalg.norm(X - centroid_per_site, axis=1)

    moran = Moran(residuals, w)
    moran_results.append({
        "Method":  method,
        "Moran's I": round(moran.I, 4),
        "E[I]": round(moran.EI, 4),
        "z-score": round(moran.z_norm, 2),
        "p-sim": round(moran.p_sim, 4),
        "Significant": moran.p_sim < 0.05,
    })
    print(f"{method}: I={moran.I:.4f}  E[I]={moran.EI:.4f}  z={moran.z_norm:.2f}  p_sim={moran.p_sim:.4f}")

moran_df = pd.DataFrame(moran_results).set_index("Method")
print("\nMoran's I summary:")
print(moran_df.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5),
                          subplot_kw={'projection': ccrs.AlbersEqualArea(
                              central_longitude=-96, central_latitude=37.5)})

for ax, (method, cl_col) in zip(axes, [("SKATER", "cl_skater"), ("Ward", "cl_ward")]):
    ax.set_extent(CONUS_EXTENT, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND,  facecolor='#f5f5f0', zorder=0)
    ax.add_feature(cfeature.OCEAN, facecolor='#c8e0f0', zorder=0)
    ax.add_feature(cfeature.STATES, linewidth=0.4, zorder=2, edgecolor='gray')
    ax.add_feature(cfeature.COASTLINE, linewidth=0.6, zorder=3)

    labels = df[cl_col].values
    centroids = np.array([X[labels == c].mean(axis=0)
                          for c in sorted(np.unique(labels))])
    label_to_idx = {c: i for i, c in enumerate(sorted(np.unique(labels)))}
    centroid_per_site = np.array([centroids[label_to_idx[c]] for c in labels])
    residuals = np.linalg.norm(X - centroid_per_site, axis=1)

    sc = ax.scatter(
        df.longitude, df.latitude,
        c=residuals, cmap='YlOrRd', s=10, vmin=0,
        transform=ccrs.PlateCarree(), zorder=4
    )
    row = moran_df.loc[method]
    ax.set_title(f"{method} residuals  |  Moran's I={row["Moran's I"]:.4f}  p={row['p-sim']:.4f}")
    plt.colorbar(sc, ax=ax, shrink=0.6, label='Distance to centroid')

rplot.panel_labels(list(axes))
plt.suptitle("Within-cluster residuals — spatial autocorrelation check", fontsize=10)
plt.tight_layout()
if SAVEFIG:
    plt.savefig(OUT_DIR / 'moran_residuals.png', dpi=200, bbox_inches='tight')
plt.show()

---
# Section 3: k-NN Sensitivity Sweep for SKATER

Sweep k=3..30 for SKATER (AEP features, n_clusters=8, floor=30).  
Results are cached to `skater_sweep_aep_k{knn:02d}.pkl` — set `FORCE_RERUN=True` to re-run.

In [ ]:
def get_or_run_skater(knn, gdf_in, attrs, n_clusters, floor, cache_dir, force=False):
    cache_path = cache_dir / f"skater_sweep_aep_k{knn:02d}.pkl"
    if cache_path.exists() and not force:
        return joblib.load(cache_path)

    w_knn = libpysal.weights.KNN.from_dataframe(gdf_in, k=knn)
    n_comp, _ = connected_components(w_knn.sparse, directed=False)
    if n_comp > 1:
        print(f"  k={knn}: disconnected graph ({n_comp} components) — skipping")
        return None

    try:
        model = Skater(gdf_in, w_knn, attrs_name=attrs, n_clusters=n_clusters, floor=floor)
        model.solve()
        result = {'labels': model.labels_, 'n_comp': n_comp, 'knn': knn}
    except Exception as e:
        print(f"  k={knn}: SKATER failed — {e}")
        return None

    joblib.dump(result, cache_path)
    return result

In [ ]:
# Reference labels for ARI comparison (k=22, the default configuration)
ref_labels = df["cl_skater"].values - 1   # 0-indexed for consistency with model.labels_

sweep_records = []

for knn in range(KNN_SWEEP_MIN, KNN_SWEEP_MAX + 1):
    print(f"k={knn:2d}...", end=' ', flush=True)
    result = get_or_run_skater(knn, gdf, RP_SCALED, N_CLUSTERS, FLOOR, OUT_DIR, FORCE_RERUN)
    if result is None:
        continue

    labels = result['labels']
    sizes  = np.bincount(labels)
    sizes  = sizes[sizes > 0]  # drop empty clusters

    rec = {
        'knn':            knn,
        'silhouette':     silhouette_score(X, labels),
        'calinski':       calinski_harabasz_score(X, labels),
        'davies_bouldin': davies_bouldin_score(X, labels),
        'ari_vs_k22':     adjusted_rand_score(ref_labels, labels),
        'min_size':       int(sizes.min()),
        'max_size':       int(sizes.max()),
        'size_ratio':     round(sizes.max() / sizes.min(), 2),
        'n_comp':         int(result['n_comp']),
    }
    sweep_records.append(rec)
    print(f"sil={rec['silhouette']:.4f}  ari={rec['ari_vs_k22']:.3f}")

sweep_df = pd.DataFrame(sweep_records)
sweep_df.to_csv(OUT_DIR / 'knn_sweep_metrics.csv', index=False)
print(f"\nSaved knn_sweep_metrics.csv  ({len(sweep_df)} rows)")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

metrics = [
    ('silhouette',     'Silhouette score',             True),
    ('calinski',       'Calinski-Harabász index',       True),
    ('davies_bouldin', 'Davies-Bouldin index',          False),
    ('ari_vs_k22',     'ARI vs k=22 reference',         True),
    ('size_ratio',     'Max/min cluster size ratio',    False),
    ('n_comp',         'Connected components (graph)', False),
]

for ax, (col, ylabel, higher_is_better) in zip(axes.flat, metrics):
    ax.plot(sweep_df.knn, sweep_df[col], marker='o', markersize=4,
            color=rplot.OKABE_ITO[1])
    ax.axvline(K_NEIGHBORS, color='gray', linestyle='--', linewidth=1,
               label=f'k={K_NEIGHBORS} (reference)')
    ax.set_xlabel('k (nearest neighbors)')
    ax.set_ylabel(ylabel)
    ax.set_title(ylabel, fontsize=9)
    ax.legend(fontsize=7)

rplot.panel_labels(list(axes.flat))
plt.suptitle(f'SKATER k-NN sensitivity sweep  (AEP features, n_clusters={N_CLUSTERS}, floor={FLOOR})',
             fontsize=10)
plt.tight_layout()
if SAVEFIG:
    plt.savefig(OUT_DIR / 'knn_sweep_metrics.png', dpi=200, bbox_inches='tight')
plt.show()

# Identify stability plateau (silhouette change < 0.005 per unit k)
sil = sweep_df['silhouette'].values
knn_vals = sweep_df['knn'].values
stable = np.where(np.abs(np.diff(sil)) < 0.005)[0]
if len(stable) > 0:
    print(f"Stability plateau (Δsilhouette < 0.005): k={knn_vals[stable[0]]}..{knn_vals[stable[-1]+1]}")
else:
    print("No clear stability plateau found at threshold 0.005")

In [ ]:
# CONUS maps at 4 selected k values
knn_show = [4, 10, 15, 22]
knn_show = [k for k in knn_show if k <= KNN_SWEEP_MAX]

fig = plt.figure(figsize=(20, 5 * len(knn_show) // 2))
n_cols = 2
n_rows = (len(knn_show) + 1) // 2

for i, knn in enumerate(knn_show, 1):
    cache_path = OUT_DIR / f"skater_sweep_aep_k{knn:02d}.pkl"
    if not cache_path.exists():
        continue
    result = joblib.load(cache_path)
    labels_map = result['labels'] + 1  # 1-indexed
    n_cl = len(np.unique(labels_map))
    colors = rplot.cluster_cmap(n_cl).colors

    ax = make_conus_ax(fig, pos=(n_rows, n_cols, i),
                       title=f'SKATER k={knn} neighbors  (sil={sweep_df.loc[sweep_df.knn==knn, "silhouette"].values[0]:.4f})')
    for cl in sorted(np.unique(labels_map)):
        idx = labels_map == cl
        ax.scatter(
            df.loc[idx, "longitude"], df.loc[idx, "latitude"],
            s=8, color=colors[cl - 1], zorder=4,
            transform=ccrs.PlateCarree(), label=f'C{cl}'
        )
    ax.legend(markerscale=2, fontsize=6, ncol=2, loc='lower left', framealpha=0.8)

plt.suptitle('SKATER cluster maps at selected k-NN values', fontsize=10)
plt.tight_layout()
if SAVEFIG:
    plt.savefig(OUT_DIR / 'knn_sweep_maps.png', dpi=200, bbox_inches='tight')
plt.show()

---
# Summary Table

In [ ]:
summary = validity_df.copy()
summary.insert(0, "k-NN", [K_NEIGHBORS, "—"])
summary.insert(0, "Features", "AEP thresholds")
summary.insert(1, "Moran's I (resid)", [
    moran_df.loc["SKATER", "Moran's I"],
    moran_df.loc["Ward", "Moran's I"]
])
summary.insert(2, "Moran's I p-sim", [
    moran_df.loc["SKATER", "p-sim"],
    moran_df.loc["Ward", "p-sim"]
])

print("=" * 80)
print("FINAL SUMMARY: SKATER vs Ward Hierarchical Clustering")
print("=" * 80)
print(summary.to_string())
print()
print("k-NN sweep (AEP features):")
print(sweep_df.to_string(index=False))